# FR labour-supply pipeline — driving the real `dclaborsupply` package

This notebook calls the **actual package API** (functions taken from the package source, not inferred). It does three things, in order:

1. **Oracle A — reproduce the certified negLL** from the stored engine-ready data via `compute_index`. Confirms the spec + JAX engine + engine-ready parquet are byte-intact. Pure package, fast, needs no EUROMOD runtime.
2. **Oracle B — the EUROMOD drift check** via the real `EuromodConnector`. The only question that actually tests whether today's pricing reproduces the stored `ils_dispy`.
3. **Re-estimation path** (`RUROModel.fit`) — only if B fails.

### Tag convention
- **`# CONFIRM`** — a file path or a data-dependent name (FR system/dataset string, column names). These are the genuine unknowns; everything else is real package API.
- Untagged calls are taken directly from the package source. They are not guesses.

### What each oracle decides
Oracle A reads `c_norm` baked into the engine-ready parquet, so it passes **even if EUROMOD drifted** — it validates the estimation path, not pricing. Oracle B re-prices through the connector, so it is the real drift test. Run A first (cheap), then B.

## 0. Environment + real package imports

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)

import dclaborsupply as dcl
print('dclaborsupply', dcl.__version__)
print('public API   :', dcl.__all__)

# real API surfaces (verified against package source)
from dclaborsupply.spec.parser import EstimationSpec, load_custom_initial_values
from dclaborsupply.data.loader import load_engine_ready_stem, load_singles, load_couples
from dclaborsupply.likelihood.index import compute_index
from dclaborsupply.models import RUROModel, Result
# EUROMOD connector lives in the APP package, imported lazily (no .NET until .run())
from dclaborsupply_app.euromod import EuromodConnector, EuromodPricingRunner

dclaborsupply 0.1.0
public API   : ['__version__', 'EstimationSpec', 'Result', 'RUMModel', 'RUROModel']


In [2]:
# CONFIRM every path in this cell against your MNL repo / storage.
# The certified FR spec (47 free params) lives in MNL, not in the package configs/.
FR_SPEC    = Path('C:/Users/hisham/Desktop/Nizam_Hisham/MNL/scripts/bpool/specs/estimation_spec_joint_pooled_v1_bll0_tlmpin.yaml')  # CONFIRM


In [3]:
# Engine-ready bundle stem: the loader reads <stem>__singles.parquet, <stem>__couples.parquet,
# and <stem>__mnlmeta.json. Point this at the certified FR engine-ready bundle.
FR_STEM    = Path('.../fr_engine_ready_pooled_v1')                                          # CONFIRM
THETA_CSV  = Path('.../theta_hat_realdata_901_v1.csv')                                      # CONFIRM
# EUROMOD model root: exactly what you pass to em.Model(...).
MODEL_ROOT = Path(r'C:\\Users\\hisham\\MNL\\EUROMOD-STORAGE\\...')                          # CONFIRM

CERT_NEGLL = 238504.6360973987   # never-move regression oracle
print('spec exists :', FR_SPEC.exists())
print('theta exists:', THETA_CSV.exists())

spec exists : False
theta exists: False


## 1. Oracle A — reproduce the certified negLL (pure package)

`compute_index` sums singles-male + singles-female + couples through the lifted JAX engine and returns the joint negLL. If this equals `238504.6360973987`, the spec, the engine, and the engine-ready data are all intact.

**Required:** `hours_band_policy="legacy_certified"`. The default `"assembled"` reads the on-disk `working_pt1` column, which disagrees with the legacy re-derivation on ~18.4k rows (~859 nats at theta_hat); the certified figure reproduces **only** under `legacy_certified` (per `loader.py`).

In [ ]:
spec = EstimationSpec.from_yaml(str(FR_SPEC))
print('free parameters:', len(spec.all_param_names))
print('wage_spec      :', spec.wage_spec, '| fixed_params:', spec.fixed_params)

# legacy_certified is REQUIRED to reproduce the FR certified negLL (see markdown above)
sm, sf, cou = load_engine_ready_stem(FR_STEM, spec, hours_band_policy='legacy_certified')
print('groups -> singles_male n_groups:', sm.n_groups,
      '| singles_female:', sf.n_groups, '| couples:', cou.n_groups)

In [ ]:
# Load theta_hat as a name->value dict (parser helper) and align to spec order.
theta_map = load_custom_initial_values(THETA_CSV)
missing = [n for n in spec.all_param_names if n not in theta_map]
assert not missing, f'theta_hat CSV missing params: {missing}'
theta_hat = np.array([theta_map[n] for n in spec.all_param_names], dtype=float)
print('theta_hat aligned, length:', len(theta_hat))

In [ ]:
negll = compute_index(spec, (sm, sf, cou), theta_hat, ruro=True, backend='jax')
diff = abs(negll - CERT_NEGLL)
print(f'negLL (package) : {negll:.7f}')
print(f'certified       : {CERT_NEGLL:.7f}')
print(f'|diff|          : {diff:.3e}')
print()
if diff < 1e-6:
    print('>>> ORACLE A PASS. Spec + engine + engine-ready data are byte-intact.')
    print('>>> The estimation path reproduces the certified figure exactly.')
else:
    print('>>> ORACLE A FAIL. Before suspecting EUROMOD, check, in order:')
    print('>>>   1. hours_band_policy is "legacy_certified" (assembled is ~859 nats off);')
    print('>>>   2. FR_STEM points at the certified bundle (right __mnlmeta.json);')
    print('>>>   3. theta_hat CSV is theta_hat_realdata_901_v1 (not a re-fit).')

## 2. Oracle B — the EUROMOD drift check (real connector)

This is the only test that asks whether **today's** EUROMOD reproduces the **stored** `ils_dispy`. We price a sample of the REAL FR baseline (the EU-SILC input you feed EUROMOD) through `EuromodConnector` now and compare to the stored baseline pricing.

**Note on the old write-back bug:** `EuromodConnector.run` returns `sim.outputs[0]` *whole* (all ~388 columns), and `EuromodPricingRunner` keeps the full output. So in this package `bsa00_s` and every means-tested column are real by construction — the 5-column truncation was the OLD MNL chunk runner, not this connector. Price through the package and that defect cannot recur.

In [ ]:
# CONFIRM the FR system/dataset strings and the baseline input/output paths.
FR_COUNTRY = 'FR'
FR_SYSTEM  = 'FR_2015'        # CONFIRM exact system name as the connector sees it
FR_DATASET = 'FR_2016_a3'     # CONFIRM exact dataset name

# The REAL FR baseline microdata in EUROMOD INPUT form (the EU-SILC input you priced c0 from):
FR_BASELINE_INPUT  = Path('.../fr_baseline_euromod_input.parquet')   # CONFIRM
# The stored baseline OUTPUT from the original pricing (for comparison):
FR_BASELINE_PRICED = Path('.../fr_baseline_priced_c0.parquet')       # CONFIRM

base_in  = pd.read_parquet(FR_BASELINE_INPUT)
stored   = pd.read_parquet(FR_BASELINE_PRICED)
# small, fast sample: first ~200 households' input rows
sample_ids = base_in['idhh'].drop_duplicates().head(200)            # CONFIRM id col
df_in = base_in[base_in['idhh'].isin(sample_ids)].copy()
print('input sample rows:', df_in.shape)

In [ ]:
# Real connector call (verified against connector.py):
#   EuromodConnector(model_root).run(df, country=, system=, dataset=) -> PricingConnectorResult
#   result.output is the FULL sim.outputs[0]; result.warnings / .errors are message strings.
conn = EuromodConnector(str(MODEL_ROOT))
res = conn.run(df_in, country=FR_COUNTRY, system=FR_SYSTEM, dataset=FR_DATASET)
fresh = res.output
print('fresh output cols:', fresh.shape[1], '  (full output, not 5)')
print('ils_dispy present:', 'ils_dispy' in fresh.columns)
if res.warnings:
    print('warnings:', res.warnings[:3])

In [ ]:
# Compare fresh ils_dispy vs stored ils_dispy on the same person rows.
key = ['idhh', 'idperson'] if 'idperson' in fresh.columns else ['idhh']   # CONFIRM keys
cmp = (stored[key + ['ils_dispy']]
       .merge(fresh[key + ['ils_dispy']], on=key, suffixes=('_stored', '_fresh')))
cmp['abs_diff'] = (cmp['ils_dispy_stored'] - cmp['ils_dispy_fresh']).abs()
print('rows compared     :', len(cmp))
print('max abs diff (EUR):', cmp['abs_diff'].max())
print('n rows > 0.01 EUR :', int((cmp['abs_diff'] > 0.01).sum()))
print()
if cmp['abs_diff'].max() <= 0.01:
    print('>>> ORACLE B PASS. No EUROMOD drift. theta_hat and the certified negLL STAND.')
    print('>>> The earlier 184-EUR finding was the stale carry-over columns, not the rules.')
    print('>>> Action: none for the baseline; just price through the package going forward.')
else:
    print('>>> ORACLE B FAIL. Today\'s connector does not reproduce stored ils_dispy.')
    print('>>> Localize: print em.__version__ + MODEL_ROOT contents; a changed connector')
    print('>>> version or model bundle is the suspect, NOT necessarily the tax rules.')
cmp.sort_values('abs_diff', ascending=False).head(10)

## 3. The fork

| Oracle A | Oracle B | Meaning | Action |
|---|---|---|---|
| PASS | PASS | Nothing moved. | theta_hat, negLL, and the W1/W4/W6 spread all stand. Resume the welfare layer. |
| PASS | FAIL | Stored data intact; today's pricing differs. | Localize the connector/bundle cause (cell below). Re-price + re-estimate **only** if the rules genuinely changed. |
| FAIL | — | Reproduction broken upstream of pricing. | Fix `legacy_certified` / stem / theta first; do not touch EUROMOD until A passes. |

Oracle A is the cheap one and isolates the estimation path; run it before spending any EUROMOD-runtime time on B.

In [ ]:
# (only if Oracle B FAILED) localize the cause — connector version vs model bundle vs rules
import euromod as em
print('euromod version :', getattr(em, '__version__', '(no __version__)'))
print('euromod file    :', getattr(em, '__file__', '(unknown)'))
print('model root      :', MODEL_ROOT)
# A jump in fresh.shape[1] vs the stored baseline's genuine EUROMOD column count is a
# connector/bundle fingerprint, not a rules edit you made.

## 4. The FR prep gap (honest note)

The package's `dclaborsupply_app.france` modules are still placeholders (`raise NotImplementedError`). The **Germany** adapter (`dclaborsupply_app.de`: `data_prep`, `draws_prep`, `pricing`, `engine_ready`) is the fully-built, tested template. So FR raw->engine-ready prep is still your MNL pipeline, **or** a port that mirrors the DE adapter.

What is already country-general and works for FR today (used above): `load_engine_ready_stem`, `compute_index`, `EuromodConnector`, `RUROModel.fit`. For full FR re-pricing through the package runner you'd also need an FR `EarningsMutationPolicy` (the yem00/yemxp 35h split) implementing the same interface as `de_earnings_policy` — that is the one FR-specific piece not yet written.

## 5. Re-estimation path (only if Oracle B fails AND the rules genuinely changed)

If pricing truly changed, re-price the engine-ready data (Two-M is moot here — the package keeps all columns), then re-fit with the SAME spec + machinery. The new theta_hat' re-certifies against a **new** oracle, never against 238504.636.

In [ ]:
# Re-fit is one call (verified against models.py). Long pole = wall time, not complexity.
#   result = RUROModel.from_spec(spec).fit((sm, sf, cou), backend='jax', compute_se=True)
#   print(result.summary())             # neg_ll, converged, hessian_min_eig, block sizes
#   print('negLL_prime:', result.convergence['neg_ll'])   # the NEW certified anchor
# Gate: clean convergence + reproduces itself on a re-run. Do NOT expect 238504.636.
print('Re-estimation is gated on Oracle B failing with a confirmed rules change. Hold otherwise.')